In [ ]:
# pip install -U transformers datasets peft accelerate bitsandbytes
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Küçük demo veri seti
ds = load_dataset("tatsu-lab/alpaca", split="train[:1%]")
def fmt(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n### Input:\n{ex.get('input','')}\n### Response:\n{ex['output']}"}
ds = ds.map(fmt, remove_columns=ds.column_names)
tok_ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=1024),
                batched=True, remove_columns=["text"])

# LoRA konfigürasyonu
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Eğitim ayarları
args = TrainingArguments(
    output_dir="out-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=200,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()

# Test
prompt = "List three benefits of PEFT in LLM fine-tuning."
x = tokenizer(prompt, return_tensors="pt").to(model.device)
y = model.generate(**x, max_new_tokens=64)
print(tokenizer.decode(y[0], skip_special_tokens=True))


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Lütfen Hugging Face token'ınızı Colab secrets'a ekleyin ve HF_TOKEN olarak adlandırın.
# Sol paneldeki "🔑" simgesine tıklayarak secret'ları yönetebilirsiniz.
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Hugging Face'e başarıyla giriş yapıldı!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN Colab secret'larına eklenmemiş. Lütfen ekleyin.")
except Exception as e:
    print(f"Hugging Face'e girişte bir hata oluştu: {e}")

In [ ]:
# pip install -U transformers datasets peft accelerate bitsandbytes
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Büyük model (donanımınıza göre erişim gerektirebilir)
base_model = "meta-llama/Llama-2-7b-hf"  # 7B model örneği

# 4-bit quantization ayarları
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Modeli 4-bit yükleme
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_cfg,
    device_map="auto"
)

# 4-bit eğitim hazırlığı
model = prepare_model_for_kbit_training(model)

# LoRA konfigürasyonu (QLoRA'nın LoRA kısmı)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]  # LLaMA benzeri
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Küçük demo veri seti
dataset = load_dataset("tatsu-lab/alpaca", split="train[:1%]")
def format_sample(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n### Input:\n{ex.get('input','')}\n### Response:\n{ex['output']}"}
dataset = dataset.map(format_sample, remove_columns=dataset.column_names)
tok_ds = dataset.map(lambda b: tokenizer(b["text"], truncation=True, max_length=1024),
                     batched=True, remove_columns=["text"])

# Eğitim parametreleri
args = TrainingArguments(
    output_dir="out-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),
    logging_steps=10,
    save_steps=200,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Trainer
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()

# Adapter'ı kaydet (küçük dosya)
model.save_pretrained("out-qlora-adapter")
tokenizer.save_pretrained("out-qlora-adapter")


In [ ]:
# pip install -U transformers datasets peft accelerate bitsandbytes
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import AdaLoraConfig, get_peft_model

# Taban model (donanımınıza göre seçin)
base_model = "microsoft/phi-2"  # 2.7B parametreli küçük model

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Demo veri seti
dataset = load_dataset("tatsu-lab/alpaca", split="train[:1%]")
def fmt(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n### Input:\n{ex.get('input','')}\n### Response:\n{ex['output']}"}
dataset = dataset.map(fmt, remove_columns=dataset.column_names)
tok_ds = dataset.map(lambda b: tokenizer(b["text"], truncation=True, max_length=1024),
                     batched=True, remove_columns=["text"])

# AdaLoRA konfigürasyonu
adalora_cfg = AdaLoraConfig(
    init_r=12, target_r=4,
    tinit=50, tfinal=200, deltaT=10,
    beta1=0.85, beta2=0.85,
    lora_alpha=32, lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]  # Mimariye göre değişebilir
)

# Model yükleme
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

# AdaLoRA adaptör ekleme
model = get_peft_model(model, adalora_cfg)
model.print_trainable_parameters()

# Eğitim ayarları
args = TrainingArguments(
    output_dir="out-adalora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=1e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=200,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Trainer
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()


In [ ]:
!pip install -U huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Lütfen Hugging Face token'ınızı Colab secrets'a ekleyin ve HF_TOKEN olarak adlandırın.
# Sol paneldeki "🔑" simgesine tıklayarak secret'ları yönetebilirsiniz.
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Hugging Face'e başarıyla giriş yapıldı!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN Colab secret'larına eklenmemiş. Lütfen ekleyin.")
except Exception as e:
    print(f"Hugging Face'e girişte bir hata oluştu: {e}")